# 🎙️ Clonador de Voz con F5-TTS
### Generador de audio para videos de terror — **Totalmente GRATIS con Google Colab**

---

## 📋 Instrucciones ANTES de empezar:

1. **Activa la GPU:** Ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → selecciona **GPU T4** → Guardar
2. **Ejecuta las celdas en orden** haciendo clic en el botón ▶️ de cada celda
3. **Ten listo:** Un audio de tu voz de al menos 10-30 segundos (limpio, sin música de fondo)

---
⚠️ **IMPORTANTE:** El audio de referencia debe ser claro, sin ruido de fondo ni música. Si lo grabas específicamente para esto, mejor.

In [ ]:
# ============================================================
# CELDA 1: Verificar GPU y preparar el entorno
# ▶️ Ejecuta esta celda primero. Tarda ~2 minutos.
# ============================================================

import subprocess
import sys

# Verificar GPU
print('🔍 Verificando GPU...')
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU detectada: {result.stdout.strip()}')
else:
    print('❌ NO se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')
    print('   Sin GPU el proceso tardará 30-60 minutos por minuto de audio.')

print()
print('📦 Instalando F5-TTS y dependencias... (esto tarda ~3-5 minutos la primera vez)')
print('   Por favor espera sin cerrar la pestaña...')
print()

subprocess.run([sys.executable, '-m', 'pip', 'install', 'f5-tts', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'soundfile', 'pydub', '-q'], check=True)

print()
print('✅ ¡Todo instalado correctamente!')
print('   Continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 2: Subir tu audio de referencia (TU VOZ)
# ▶️ Ejecuta esta celda y sube tu archivo de audio
# ============================================================

from google.colab import files
import shutil
import os

print('📁 Se abrirá un selector de archivos...')
print('   Sube un archivo de audio con tu voz (.mp3 o .wav)')
print('   Requisitos: mínimo 10 segundos, sin música de fondo\n')

uploaded = files.upload()

if not uploaded:
    print('❌ No subiste ningún archivo. Vuelve a ejecutar esta celda.')
else:
    ref_audio_original = list(uploaded.keys())[0]
    ref_audio_path = '/content/mi_voz_referencia.wav'

    if ref_audio_original.lower().endswith('.mp3'):
        print('🔄 Convirtiendo MP3 a WAV...')
        from pydub import AudioSegment
        audio = AudioSegment.from_mp3(ref_audio_original)
        audio = audio.set_channels(1).set_frame_rate(24000)
        audio.export(ref_audio_path, format='wav')
        print('✅ Convertido a WAV correctamente.')
    else:
        shutil.copy(ref_audio_original, ref_audio_path)
        print(f'✅ Audio de referencia guardado.')

    print()
    print(f'🎙️ Archivo de referencia listo: {ref_audio_path}')
    print('   Continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 3: Escribe qué dice tu audio de referencia
# ▶️ Cambia el texto y ejecuta
# ============================================================

# ⚠️ Escribe aquí EXACTAMENTE lo que dices en el audio que subiste

TEXTO_DEL_AUDIO_REFERENCIA = """Escribe aquí exactamente lo que dices en tu audio de referencia.
Por ejemplo: Hola, soy Nelson y este es mi canal de terror donde exploramos los misterios más oscuros del mundo."""

# -------------------------------------------------------

print('📝 Texto de referencia configurado:')
print(f'   "{TEXTO_DEL_AUDIO_REFERENCIA[:120]}..."' if len(TEXTO_DEL_AUDIO_REFERENCIA) > 120 else f'   "{TEXTO_DEL_AUDIO_REFERENCIA}"')
print()
print('✅ Si el texto es correcto, continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 4: ✍️ ESCRIBE TU GUIÓN AQUÍ
# ▶️ Pega el guion y ejecuta
# ============================================================

GUION_A_NARRAR = """Bienvenidos a otro video donde la realidad supera a la ficción.
Esta es la historia de una casa abandonada en las afueras de la ciudad, donde los vecinos
aseguran haber escuchado voces en la oscuridad. Nadie sabe qué ocurrió realmente aquella noche.
Pero lo que está a punto de descubrir cambiará su perspectiva para siempre."""

# -------------------------------------------------------

print('📖 Guion a narrar:')
print('-' * 60)
print(GUION_A_NARRAR)
print('-' * 60)
palabras = len(GUION_A_NARRAR.split())
print(f'   Palabras: {palabras} | Duración estimada: ~{palabras // 130} min {(palabras % 130) * 60 // 130} seg')
print()
print('✅ Si el guion es correcto, continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 5: 🚀 GENERAR AUDIO CON TU VOZ
# ============================================================
#
# 🎛️ CONTROL DE VELOCIDAD:
# Cambia el número de VELOCIDAD_NARRADOR:
#   0.7 = Muy lento (dramático, terror intenso)
#   0.8 = Lento (recomendado para terror)
#   0.9 = Ligeramente lento
#   1.0 = Normal (velocidad del audio de referencia)
#   1.2 = Rápido
#
# Si salió MUY RÁPIDO → pon 0.7 o 0.8
# ============================================================

VELOCIDAD_NARRADOR = 0.8  # ← CAMBIA ESTE NÚMERO

# -------------------------------------------------------

import time
import soundfile as sf
import numpy as np
from pydub import AudioSegment

OUTPUT_PATH = '/content/audio_generado.wav'
OUTPUT_AJUSTADO = '/content/audio_final.wav'

print(f'🎙️ Generando audio con tu voz...')
print(f'   Velocidad configurada: {VELOCIDAD_NARRADOR}x')
print('   Cargando modelo (solo la primera vez tarda ~2 min)...')
print()

try:
    from f5_tts.api import F5TTS

    f5tts = F5TTS()
    print('✅ Modelo cargado.')
    print('🔄 Generando audio...')
    start_time = time.time()

    wav, sr, spect = f5tts.infer(
        ref_file=ref_audio_path,
        ref_text=TEXTO_DEL_AUDIO_REFERENCIA,
        gen_text=GUION_A_NARRAR,
        file_wave=OUTPUT_PATH,
        speed=VELOCIDAD_NARRADOR,
        seed=42
    )

    elapsed = time.time() - start_time
    print(f'✅ ¡Audio generado en {elapsed:.1f} segundos!')

    # Verificar duración
    data, sample_rate = sf.read(OUTPUT_PATH)
    duracion = len(data) / sample_rate
    print(f'   Duración del audio: {duracion:.1f} segundos ({duracion/60:.1f} minutos)')
    print()
    print('✅ Continúa con la Celda 6 para escuchar y descargar ▼')

except Exception as e:
    print(f'❌ Error: {e}')
    print()
    print('💡 Si el error dice "ref_audio_path no definido", ejecuta primero las Celdas 2 y 3.')
    print('   Si el error es de memoria CUDA, el guion es muy largo. Divídelo en partes (ver Celda Extra).')

In [ ]:
# ============================================================
# CELDA 6: 🔊 ESCUCHAR Y DESCARGAR EL AUDIO
# ▶️ Ejecuta para reproducir y descargar tu audio
# ============================================================

import IPython.display as ipd
from google.colab import files
import os

if os.path.exists(OUTPUT_PATH):
    print('🔊 Reproduciendo audio generado...')
    print(f'   Velocidad usada: {VELOCIDAD_NARRADOR}x')
    print()
    display(ipd.Audio(OUTPUT_PATH))
    print()
    print('📥 Descargando archivo...')
    files.download(OUTPUT_PATH)
    print('✅ ¡Listo! El archivo se descargó como "audio_generado.wav"')
    print()
    print('💡 Si todavía suena rápido → vuelve a la Celda 5 y baja VELOCIDAD_NARRADOR a 0.7')
    print('   Si suena bien → ¡úsalo en tu video de terror! 🎬')
else:
    print('❌ No se encontró el archivo. Ejecuta primero la Celda 5.')

In [ ]:
# ============================================================
# CELDA EXTRA A: 🐢 Ajustar velocidad SIN regenerar
# Úsala si el audio ya se generó pero quieres hacerlo más lento/rápido
# SIN tener que esperar a que F5-TTS genere de nuevo
# ============================================================

from pydub import AudioSegment
import IPython.display as ipd
from google.colab import files

# ¿Cuánto más lento quieres hacerlo?
# 0.8 = 20% más lento | 0.7 = 30% más lento | 0.6 = 40% más lento
FACTOR_VELOCIDAD = 0.8  # ← CAMBIA ESTE NÚMERO

# -------------------------------------------------------

print(f'🐢 Ajustando velocidad del audio al {int(FACTOR_VELOCIDAD*100)}%...')

audio = AudioSegment.from_wav(OUTPUT_PATH)

# Cambiar velocidad sin cambiar el tono
audio_slow = audio._spawn(audio.raw_data, overrides={
    'frame_rate': int(audio.frame_rate * FACTOR_VELOCIDAD)
}).set_frame_rate(audio.frame_rate)

OUTPUT_LENTO = '/content/audio_mas_lento.wav'
audio_slow.export(OUTPUT_LENTO, format='wav')

duracion_orig = len(audio) / 1000
duracion_nueva = len(audio_slow) / 1000
print(f'   Duración original:  {duracion_orig:.1f} seg')
print(f'   Duración ajustada:  {duracion_nueva:.1f} seg')
print()

print('🔊 Escucha el resultado:')
display(ipd.Audio(OUTPUT_LENTO))
print()
files.download(OUTPUT_LENTO)
print('📥 Descargado como "audio_mas_lento.wav"')
print()
print('💡 Si aún está rápido, baja más el FACTOR_VELOCIDAD (ej: 0.65) y ejecuta de nuevo.')

In [ ]:
# ============================================================
# CELDA EXTRA B: Generar en PARTES para guiones largos
# 💡 Úsala si tu guion es muy largo (+5 minutos de audio)
# ============================================================

import os
import soundfile as sf
import numpy as np
from f5_tts.api import F5TTS
from google.colab import files

VELOCIDAD_PARTES = 0.8  # ← Misma velocidad que en Celda 5

# Tu guion dividido en párrafos
GUION_EN_PARTES = [
    """Esta es la primera parte del guion. Bienvenidos al canal.""",
    """Esta es la segunda parte. El misterio comienza aquí.""",
    """Esta es la tercera parte. El clímax de la historia.""",
]

f5tts = F5TTS()
partes_audio = []
sample_rate = 24000

for i, parte in enumerate(GUION_EN_PARTES):
    print(f'🔄 Generando parte {i+1}/{len(GUION_EN_PARTES)}...')
    output_parte = f'/content/parte_{i+1}.wav'

    wav, sr, _ = f5tts.infer(
        ref_file=ref_audio_path,
        ref_text=TEXTO_DEL_AUDIO_REFERENCIA,
        gen_text=parte,
        file_wave=output_parte,
        speed=VELOCIDAD_PARTES,
        seed=42
    )

    data, sr = sf.read(output_parte)
    partes_audio.append(data)
    print(f'   ✅ Parte {i+1} lista.')

# Silencio de 0.5 segundos entre partes
silencio = np.zeros(int(sample_rate * 0.5))

audio_final = []
for i, parte in enumerate(partes_audio):
    audio_final.append(parte)
    if i < len(partes_audio) - 1:
        audio_final.append(silencio)

audio_completo = np.concatenate(audio_final)
OUTPUT_COMPLETO = '/content/guion_completo.wav'
sf.write(OUTPUT_COMPLETO, audio_completo, sample_rate)

print()
print(f'✅ Guion completo: {len(audio_completo)/sample_rate:.1f} segundos')

import IPython.display as ipd
display(ipd.Audio(OUTPUT_COMPLETO))
files.download(OUTPUT_COMPLETO)
print('📥 Descarga iniciada: guion_completo.wav')

---
## 🆘 Solución de problemas

| Problema | Solución |
|---|---|
| **Audio muy rápido** | En Celda 5, baja `VELOCIDAD_NARRADOR` a `0.7` o usa la **Celda Extra A** |
| **Audio robótico/raro** | El texto de referencia no coincide exactamente con el audio |
| **`CUDA out of memory`** | Guion muy largo → usa la **Celda Extra B** para dividirlo |
| **`ref_audio_path not defined`** | Ejecuta primero la **Celda 2** |
| **No se detectó GPU** | `Entorno de ejecución` → `Cambiar tipo` → **GPU T4** |

---
## 🎛️ Guía de velocidades

| Valor | Resultado | Ideal para |
|---|---|---|
| `0.6` | Muy muy lento | Narración dramática intensa |
| `0.7` | Lento | Terror oscuro, pausado |
| `0.8` | Ligeramente lento | **Recomendado para terror** |
| `1.0` | Normal | Velocidad del audio de referencia |
| `1.2` | Rápido | Acción, suspenso |

---
## 💡 Tips para mejor calidad

1. Graba 30 segundos **hablando despacio y claro**, en silencio total
2. El texto de referencia debe coincidir **al 100%** con el audio
3. Usa **puntuación correcta** en el guion (comas y puntos = pausas naturales)
4. Si el modelo no suena bien, prueba distintos audios de referencia

---
*Notebook creado para el flujo de trabajo de creación de videos de terror 🩸*